In [ ]:
# ML Libs
import torch
import torch.nn as nn
from torch.distributions.normal import Normal
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error
from torch.utils.data import Dataset, DataLoader

# File processing stuff
import os
from pathlib import Path

# STEP 1: Data loading/processing
def load_and_combine_data(positive_root, negative_root):
    """
    Recursively walks through all project folders, reads every CSV, tags it as
    'positive' or 'negative' based on its source folder, and combines them into one
    single, massive dataset for training. This is a robust and essential first step.

    Might switch to an easier path soon?
    """
    all_data_frames = []
    for root_path, label in [(positive_root, 1), (negative_root, 0)]:
        print(f"Searching in '{root_path}'...")
        # find all CSVs in all subdirectories.
        for file_path in Path(root_path).rglob('*.csv'):
            try:
                df = pd.read_csv(file_path)
                # add a 'source_label' column to keep track of the data's origin.
                # this is more for debugging
                df['source_label'] = label
                all_data_frames.append(df)
            except Exception as e:
                print(f"  - Could not process file {file_path.name}: {e}")
    print(f"\nFound and processed {len(all_data_frames)} files.")
    if not all_data_frames:
        raise ValueError("No data files were found! Check your root paths.")
    return pd.concat(all_data_frames, ignore_index=True)

# STEP 2: Model definition
# As per Professor's request, I was modifying the model in this github:
# https://github.com/davidmrau/mixture-of-experts/blob/master/moe.py
# ^^^^ This is main inspiration for the model
# https://cameronrwolfe.substack.com/p/nano-moe <-- this can also be a good read
# but i wasn't fully sure how to implement/use

# This helper class is part of the original MoE architecture
class SparseDispatcher:
    def __init__(self, num_experts, gates):
        self._gates, self._num_experts = gates, num_experts
        sorted_experts, index_sorted_experts = torch.nonzero(gates).sort(0)
        _, self._expert_index = sorted_experts.split(1, dim=1)
        self._batch_index = torch.nonzero(gates)[index_sorted_experts[:, 1], 0]
        self._part_sizes = (gates > 0).sum(0).tolist()
        gates_exp = gates[self._batch_index.flatten()]
        self._nonzero_gates = torch.gather(gates_exp, 1, self._expert_index)

    def dispatch(self, inp):
        inp_exp = inp[self._batch_index].squeeze(1)
        return torch.split(inp_exp, self._part_sizes, dim=0)

    def combine(self, expert_out, multiply_by_gates=True):
        stitched = torch.cat(expert_out, 0)
        stitched = stitched.mul(
            self._nonzero_gates) if multiply_by_gates else stitched
        zeros = torch.zeros(self._gates.size(
            0), expert_out[-1].size(1), requires_grad=True, device=stitched.device)
        combined = zeros.index_add(0, self._batch_index, stitched.float())
        return combined

# Modifying the MLP in that Sparesely gated structure from the github repo
# Added more layers for deeper learning and components
# like BatchNorm and Dropout to ensure stable, effective training on complex data.
class EnhancedMLP(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, dropout_prob=0.3):
        super(EnhancedMLP, self).__init__()
        self.net = nn.Sequential(nn.Linear(input_size, hidden_size),
                                 nn.BatchNorm1d(hidden_size), nn.LeakyReLU(
                                     inplace=True),
                                 nn.Dropout(dropout_prob), nn.Linear(
                                     hidden_size, hidden_size//2),
                                 nn.BatchNorm1d(
                                     hidden_size//2), nn.LeakyReLU(inplace=True),
                                 nn.Dropout(dropout_prob), nn.Linear(hidden_size//2, output_size))

    def forward(self, x):
        return self.net(x)

# WHY: This is the main "manager" model. Look at the data for each
# RNA design and intelligently route it to a small group of its best `EnhancedMLP` specialists.
# For this one, i was just trying things that worked best at the time and I need to refresh this more.
# Very similar to the one in the github repo
class MoE(nn.Module):
    def __init__(self, input_size, output_size, num_experts, hidden_size, noisy_gating=True, k=4):
        super(MoE, self).__init__()
        self.noisy_gating, self.num_experts, self.k = noisy_gating, num_experts, k
        self.experts = nn.ModuleList([EnhancedMLP(
            input_size, output_size, hidden_size) for _ in range(self.num_experts)])
        self.w_gate = nn.Parameter(torch.zeros(
            input_size, num_experts), requires_grad=True)
        self.w_noise = nn.Parameter(torch.zeros(
            input_size, num_experts), requires_grad=True)
        self.softplus, self.softmax = nn.Softplus(), nn.Softmax(1)
        self.register_buffer("mean", torch.tensor([0.0]))
        self.register_buffer("std", torch.tensor([1.0]))

    def cv_squared(self, x):
        eps = 1e-10
        return torch.tensor([0], device=x.device, dtype=x.dtype) if x.shape[0] == 1 else x.float().var() / (x.float().mean()**2+eps)

    def noisy_top_k_gating(self, x, train, noise_epsilon=1e-2):
        clean_logits = x@self.w_gate
        logits = clean_logits
        if self.noisy_gating and train:
            raw_noise_stddev = x @ self.w_noise # "@" is matrix mult operator
            noise_stddev = self.softplus(raw_noise_stddev)+noise_epsilon
            noisy_logits = clean_logits + \
                (torch.randn_like(clean_logits)*noise_stddev)
            logits = noisy_logits
        logits = self.softmax(logits)
        top_logits, top_indices = logits.topk(
            min(self.k+1, self.num_experts), dim=1)
        top_k_logits, top_k_indices = top_logits[:,
                                                 :self.k], top_indices[:, :self.k]
        top_k_gates = top_k_logits/(top_k_logits.sum(1, keepdim=True)+1e-6)
        zeros = torch.zeros_like(logits, requires_grad=True)
        gates = zeros.scatter(1, top_k_indices, top_k_gates)
        load = (gates > 0).sum(0)
        return gates, load

    def forward(self, x, loss_coef=1e-2):
        gates, load = self.noisy_top_k_gating(x, self.training)
        importance = gates.sum(0)
        loss = self.cv_squared(importance)+self.cv_squared(load)
        loss *= loss_coef
        dispatcher = SparseDispatcher(self.num_experts, gates)
        expert_inputs = dispatcher.dispatch(x)
        expert_outputs = [self.experts[i](
            expert_inputs[i]) for i in range(self.num_experts)]
        y = dispatcher.combine(expert_outputs)
        return y, loss

# STEP 3 & 4: Data set preparation and training
class RiboswitchDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.targets[idx]


if __name__ == "__main__":
    # --- Data Paths ---
    POSITIVE_DATA_ROOT = '/content/drive/MyDrive/Riboswitches/Cox Data'
    NEGATIVE_DATA_ROOT = '/content/drive/MyDrive/Riboswitches/Negative Data'

    master_df = load_and_combine_data(POSITIVE_DATA_ROOT, NEGATIVE_DATA_ROOT)

    # --- Preprocess Data ---
    print("\nPreprocessing data...")
    # final prediction target --> direct quantitative measure of the final complex's stability.
    # This is what I was told we were trying to do predict how artificial changes to RNA affect RNA complex stability.
    # Not sure if completely right, need to double check.
    TARGET_VARIABLE = 'e_complex_s'

    # Select the most relevant numerical columns (LSTM For non-numeric see below).
    # These represent the environmental conditions and high-level properties of the RNA.
    NUMERICAL_FEATURES = ['salt_molar', 'temperature',
                          'Switch_gc', 'Switch_at', 'e_switch_s', 'e_target_s']

    # We do still need the RNA sequence to predict its function.
    # So just a trivial One-Hot Encoding on these columns as a simple substitute for the LSTM (Which audrey was working on).
    # ^ I just did this to get an idea of how my model is working
    SEQUENCE_FEATURES = ['riboswitch sequence', 'target_sequence']

    # Simple Data Cleaning
    all_feature_cols = NUMERICAL_FEATURES + SEQUENCE_FEATURES
    master_df.dropna(subset=[TARGET_VARIABLE] + all_feature_cols, inplace=True)

    # Convert of all our chosen features into a single
    # numerical matrix that the model can understand.
    # It scales numbers and converts the sequence text into numerical format.
    preprocessor = ColumnTransformer(transformers=[
        ('num_scaler', StandardScaler(), NUMERICAL_FEATURES),
        ('seq_encoder', OneHotEncoder(handle_unknown='ignore',
         sparse_output=False, max_categories=20), SEQUENCE_FEATURES)
    ], remainder='drop')

    y = master_df[TARGET_VARIABLE].values
    X = preprocessor.fit_transform(master_df)

    print("Data successfully preprocessed.")
    print(f"Feature matrix shape (X): {X.shape}")

    # Split the data into a training set and a validation set.
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42)

    train_dataset = RiboswitchDataset(X_train, y_train)
    val_dataset = RiboswitchDataset(X_val, y_val)
    train_loader = DataLoader(dataset=train_dataset,
                              batch_size=128, shuffle=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=128)

    # --- Initialize and Train ---
    INPUT_SIZE = X.shape[1]
    OUTPUT_SIZE = 1  # Predicting a single value (energy).
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    model = MoE(input_size=INPUT_SIZE, output_size=OUTPUT_SIZE,
                num_experts=8, hidden_size=512, k=2).to(device)
    # For predicting a number, MSE is the standard error function.
    criterion = nn.MSELoss()
    # AdamW is a standard, robust, modern optimizer. <-- Might have some room for fine-tuning
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # --- Step 4: Training and Evaluation Loop ---
    # I basically used a training loop similar to another project I did
    # might need to change
    print("Starting training...")
    EPOCHS = 30
    for epoch in range(EPOCHS):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), inputs.to(device)
            optimizer.zero_grad()
            predictions, load_balancing_loss = model(inputs)
            regression_loss = criterion(
                predictions.squeeze(-1), targets.float())
            total_loss = regression_loss + load_balancing_loss
            total_loss.backward()
            optimizer.step()

        # --- Validation and Accuracy Measurement ---
        model.eval()
        all_targets, all_predictions = [], []
        with torch.no_grad():  # We don't need to calculate gradients for evaluation
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                predictions, _ = model(inputs)
                all_targets.extend(targets.cpu().numpy())
                all_predictions.extend(predictions.squeeze(-1).cpu().numpy())

        # Calculate R-squared and Mean Absolute Error to see how "accurate"
        # our model is
        r2 = r2_score(all_targets, all_predictions)
        mae = mean_absolute_error(all_targets, all_predictions)

        print(
            f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {total_loss.item():.4f} | Val R²: {r2:.3f}, Val MAE: {mae:.3f}')

    print("\nTraining finished.")